# 강의 04 · 실습 6 — 커스텀 MCP 서버 · (2-1) 빈칸 채우기 I

## 1. 문제상황

- 사내 도구 서버를 만드는 팀은 현재 시각 조회나 파일 읽기 같은 기능을 공개된 MCP 서버로 이미 붙여 쓰고 있습니다.
- 그런데 팀 안에서만 쓰는 계산 함수는 파이썬 파일 안에 함수로만 존재합니다.
- 그 함수는 이름도 인자 형식도 표준 규격 위에 올라와 있지 않아서, 모델이 도구로 부를 수 있는 대상이 아닙니다.
- 함수를 쓰려면 사람이 노트북을 열어 직접 호출하고 결과를 옮겨 적어야 합니다.

## 2. 문제와 목표

- **문제**: 직접 쓴 파이썬 함수가 표준 규격 위에 있지 않아 모델이 도구로 부르지 못합니다.
- **목표**: 정수 덧셈과 곱셈 함수 두 개를 MCP 도구로 등록한 서버를 만들고, 클라이언트가 실행 명령만으로 그 서버를 띄워 도구 목록을 받아 계산 질문에 쓰는 흐름을 만듭니다.
    - 서버: `FastMCP("math")` 인스턴스 하나에 도구 두 개(`add`·`multiply`)를 등록하고 표준입출력으로 시작하는 파일 `math_server.py`입니다.
    - 클라이언트: 서버 파일을 띄우는 연결 선언, 도구 목록 수령, 도구 호출 루프(단계 0의 `build_loop`)의 세 부분입니다.
    - 계산 질문: 「17 곱하기 4에 25를 더하면?」 한 문장이며 코드에 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 클라이언트가 받은 도구 목록에 `add`와 `multiply`가 설명과 함께 들어 있습니다.
    - 질문에 모델이 두 도구를 차례로 호출해 93이라고 답하는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex06_s1_diagram.svg)

## 4. 단계별 요구사항

1. **서버 인스턴스를 만듭니다.**
    - `FastMCP`로 이름이 `math`인 서버 객체를 만듭니다.
    - 이 줄이 서버 파일 `math_server.py`의 시작입니다.
2. **함수를 도구로 등록합니다.**
    - 정수 두 개를 더하는 `add`와 곱하는 `multiply`를 `@mcp.tool()`로 등록합니다.
    - 인자와 반환에 `int` 타입 힌트를 붙이고, 독스트링으로 설명 한 줄을 답니다.
3. **서버를 시작합니다.**
    - 파일이 직접 실행될 때만 `mcp.run(transport="stdio")`가 돌도록 `__main__` 가드 안에 넣습니다.
4. **클라이언트에서 띄워 확인합니다.**
    - `MultiServerMCPClient`에 서버 파일을 띄우는 실행 명령을 적고, `get_tools()`로 도구 목록을 받아 이름과 설명을 출력합니다.
    - 받은 도구를 단계 0의 `build_loop`에 넣고 「17 곱하기 4에 25를 더하면?」을 물어 도구 호출과 최종 답을 출력합니다.

## 5. 코드 골격 — MCP 서버 3단

FastMCP로 서버를 세우는 순서는 다음 세 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 세 단계와 하나씩 대응합니다. 서버 코드는 `%%writefile`로 파일에 쓰고, 단계 ③의 확인 셀에서 클라이언트가 그 파일을 실행 명령으로 띄웁니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 인스턴스 생성 | 서버 객체를 이름과 함께 만듭니다 | `mcp = FastMCP("math")` | 1 |
| ② 도구 등록 | 파이썬 함수에 표시를 붙이고, 타입 힌트와 설명 한 줄을 답니다 | `@mcp.tool()`, `def add(a: int, b: int) -> int` | 2 |
| ③ 서버 시작 | 표준입출력으로 말하도록 지정해 서버를 띄웁니다. 클라이언트가 실행 명령으로 띄우고 도구 목록을 받아 씁니다 | `mcp.run(transport="stdio")`, `MultiServerMCPClient`, `get_tools()` | 3, 4 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

클라이언트 쪽 준비입니다. 라이브러리를 불러오고 모델을 준비하고, 도구 호출 루프 `build_loop`와 연결 선언 함수 `server_config`를 정의합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `build_loop`는 받아 온 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결하는 도구 호출 루프입니다. 서버 코드가 아니라 서버를 쓰는 쪽의 코드입니다.
- `sys.stderr = sys.__stderr__` 줄은 노트북 전용입니다. 노트북 커널은 표준 오류 스트림을 화면용 객체로 바꿔 두는데, 서버 프로세스를 띄우는 코드는 원래의 표준 오류 스트림을 요구하므로 되돌려 놓습니다. 이 줄은 MCP 클라이언트를 불러오기 전에 있어야 합니다.
- `server_config`는 서버 파일 하나를 표준입출력으로 띄우는 연결 선언입니다. `sys.executable`은 지금 돌고 있는 파이썬 러너입니다. `FASTMCP_LOG_LEVEL`은 서버의 안내 로그가 화면을 채우지 않게 하는 설정입니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import ____, ____
from typing import Annotated, TypedDict

from langchain.chat_models import ____
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 커널의 stderr에는 fileno()가 없어 서버 프로세스 시작이 실패하므로 원래 stderr로 되돌린다
from langchain_mcp_adapters.client import ____
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ____

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_loop(tools):
    """도구 호출 루프. 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결한다."""
    bound = llm.bind_tools(tools)

    def call_model(state: State) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: State) -> str:
        return "tools" if state["messages"][-1].tool_calls else END

    g = StateGraph(State)
    g.add_node("model", call_model)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
    g.add_edge("tools", "model")
    return g.compile()


def text_of(m) -> str:
    """메시지 내용이 콘텐츠 블록 목록이면 글자 부분만 이어 붙인다."""
    if isinstance(m.content, list):
        return " ".join(p.get("text", "") for p in m.content if isinstance(p, dict))
    return str(m.content)


def show(result) -> None:
    """실행 결과의 메시지를 종류·도구 호출·상태와 함께 한 줄씩 출력한다."""
    for m in result["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif kind == "ToolMessage":
            print(f"[{kind}] status={m.status!r} {text_of(m)[:160]}")
        elif m.content:
            print(f"[{kind}] {text_of(m)[:300]}")


def server_config(file: str) -> dict:
    """서버 파일 하나를 표준입출력으로 띄우는 연결 선언을 만든다."""
    return {"command": sys.executable, "args": [str(Path(file).resolve())],
            "transport": "stdio", "env": {"FASTMCP_LOG_LEVEL": "ERROR"}}


print("클라이언트 준비를 마쳤습니다.")

### 단계 ① — 인스턴스 생성 (요구사항 1)

`FastMCP(이름)`이 서버 한 대입니다. 괄호 안 이름이 이 서버의 이름입니다. `%%writefile`이 이 셀의 내용을 서버 파일로 저장합니다. 서버 파일은 사람이 직접 실행하지 않고, 단계 ③에서 클라이언트가 띄웁니다.

In [ ]:
%%writefile math_server.py
from mcp.server.fastmcp import ____

mcp = ____

### 단계 ② — 도구 등록 (요구사항 2)

`@mcp.tool()` 한 줄을 얹으면 그 파이썬 함수가 MCP 도구가 됩니다. 함수 이름이 도구 이름, 독스트링이 도구 설명, 인자와 반환의 타입 힌트가 도구 규격으로 그대로 나갑니다. `%%writefile -a`는 서버 파일 뒤에 이어 붙입니다.

In [ ]:
%%writefile -a math_server.py

@____
def add(a: ____, b: ____) -> ____:
    """두 정수를 더한다."""
    return ____


@____
def multiply(a: ____, b: ____) -> ____:
    """두 정수를 곱한다."""
    return ____

### 단계 ③ — 서버 시작 (요구사항 3, 4)

`transport="stdio"`는 표준입출력으로 말한다는 뜻입니다. 서버는 포트를 열지 않고, 자신을 실행한 쪽과 입출력으로 주고받습니다. 아래 두 셀(③-a, ③-b)이 모두 이 단계에 속합니다.

#### 단계 ③-a — 시작 코드를 서버 파일에 붙입니다

In [ ]:
%%writefile -a math_server.py

if __name__ == "__main__":
    mcp.run(____)

#### 단계 ③-b — 클라이언트에서 띄우고 확인합니다

클라이언트에 적는 것은 서버를 띄우는 실행 명령뿐입니다. 서버 코드는 클라이언트에 등장하지 않습니다. `get_tools()`가 서버가 내놓은 도구 목록을 받아 오고, 그 목록을 `build_loop`에 그대로 넣습니다.

In [ ]:
print(Path("math_server.py").read_text(encoding="utf-8"))

client = MultiServerMCPClient({"math": ____})
tools = ____
print("서버가 준 도구:", [(t.name, t.description) for t in tools])

# 여기에 루프 연결과 질문 실행, 결과 출력을 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음을 확인합니다.

1. `%%writefile` 셀마다 `Writing math_server.py` 또는 `Appending to math_server.py`가 출력됩니다. 서버 파일이 세 셀에 나뉘어 완성됩니다.
2. 단계 ③-b의 파일 내용 출력에서 임포트, 인스턴스 생성, 도구 두 개, 시작 가드가 순서대로 들어 있습니다.
3. `서버가 준 도구:` 줄에 `add`와 `multiply`가 독스트링 그대로의 설명과 함께 있습니다. 함수 이름과 독스트링이 그대로 도구 이름과 설명이 되었습니다.
4. 메시지 기록에서 `AIMessage`의 `tool_calls`에 `multiply`와 `add`가 차례로 나오고, `ToolMessage`의 `status='success'` 뒤에 68과 93이 있고, 마지막 `AIMessage`가 93이라고 답합니다.

확인이 하나라도 다르면 `lec04_ex06_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.
